<a href="https://colab.research.google.com/github/ancestor9/Scikit-learn-model-selection-evaluation/blob/main/optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install optuna

In [2]:
# @title **1. Obtain Data & Split Data into Train & Test**
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
cancer = pd.DataFrame(np.c_[cancer['target'], cancer['data']],
                      columns = np.append('target', cancer['feature_names']))
cancer['target'] = cancer['target'].map({0.0: 1, 1.0:0}).astype('int64')

# Separate the independent variables from the dependent variable.
X = cancer.drop('target', axis = 1)
y = cancer['target']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    stratify=y,
                                                    random_state=0)

**Optuna의 기본 알고리즘인 TPE (Tree-structured Parzen Estimator)라는 베이지안 최적화(Bayesian Optimization) 기반의 탐색 방식**

In [3]:
# @title **2. Optuna Optimization**

import optuna
import sklearn.datasets
import sklearn.ensemble
import sklearn.model_selection
"""
Optuna example that optimizes a classifier configuration for Iris dataset using sklearn.

In this example, we optimize a classifier configuration for Iris dataset. Classifiers are from
scikit-learn. We optimize both the choice of classifier (among SVC and RandomForest) and their
hyperparameters.

"""
def objective(trial):
    # 1. 랜덤 포레스트의 주요 하이퍼파라미터들 정의
    n_estimators = trial.suggest_int("n_estimators", 50, 500, step=50)  # 생성할 트리의 개수
    max_depth = trial.suggest_int("max_depth", 3, 30)  # 트리의 최대 깊이 (None 대신 상한선 설정 가능)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)  # 노드를 분할하기 위한 최소 샘플 수
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)  # 리프 노드에 있어야 할 최소 샘플 수
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])  # 각 노드에서 사용할 피처 수의 방식
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])  # 부트스트랩 샘플 사용 여부
    criterion = trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"])  # 분할 품질 측정 기준

    # 2. 파라미터를 적용한 랜덤 포레스트 모델 생성
    classifier_obj = sklearn.ensemble.RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        criterion=criterion,
        random_state=42,
        n_jobs=-1
    )

    # 3. 교차 검증 점수 계산
    score = sklearn.model_selection.cross_val_score(
        classifier_obj, X, y, n_jobs=-1, cv=3
    )
    accuracy = score.mean()
    return accuracy


if __name__ == "__main__":
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100)

    # 4. 결과 확인
    print("Best Params:", study.best_params)

    # 5. **kwargs를 이용해 최적 파라미터로 최종 모델 생성 및 학습
    best_params = study.best_params
    best_rf_model = sklearn.ensemble.RandomForestClassifier(
        **best_params,
        random_state=42,
        n_jobs=-1
    )

    best_rf_model.fit(X, y)

[I 2026-11-12 04:44:01,237] A new study created in memory with name: no-name-c583044e-7a39-4854-b9aa-05fd4ceac251
[I 2026-11-12 04:44:06,791] Trial 0 finished with value: 0.9508029332590736 and parameters: {'n_estimators': 450, 'max_depth': 17, 'min_samples_split': 7, 'min_samples_leaf': 10, 'max_features': 'log2', 'bootstrap': True, 'criterion': 'gini'}. Best is trial 0 with value: 0.9508029332590736.
[I 2026-11-12 04:44:09,163] Trial 1 finished with value: 0.9543209876543209 and parameters: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'bootstrap': True, 'criterion': 'gini'}. Best is trial 1 with value: 0.9543209876543209.
[I 2026-11-12 04:44:12,913] Trial 2 finished with value: 0.9630929174788824 and parameters: {'n_estimators': 300, 'max_depth': 11, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': False, 'criterion': 'entropy'}. Best is trial 2 with value: 0.9630929174788824.
[I 2026-

Best Params: {'n_estimators': 50, 'max_depth': 13, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': False, 'criterion': 'entropy'}


In [4]:
best_rf_model

RandomForestClassifier(bootstrap=False, criterion='entropy', max_depth=13,
                       max_features='log2', min_samples_leaf=3,
                       min_samples_split=5, n_estimators=50, n_jobs=-1,
                       random_state=42)

In [5]:
# @title **예측모형의 성능지표(검증데이터) 산출 사용자정의 함수**
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    ConfusionMatrixDisplay
)


def evaluate_model(model, X_test, y_test, model_name="Model"):
    """
    모델과 테스트 데이터를 받아 주요 성능 지표를 계산하고
    컬럼 형태의 Pandas DataFrame으로 반환하는 함수
    """
    # 1. 예측값 및 확률값 계산
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # 2. 혼동 행렬 및 오류 개수 계산
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    total_errors = fp + fn  # 오탐(틀린 개수) 총합

    # 3. 주요 지표 계산
    metrics = {
        "Accuracy (정확도)": accuracy_score(y_test, y_pred),
        "Precision (정밀도)": precision_score(y_test, y_pred),
        "Recall (재현율)": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "False Positives (FP)": int(fp),
        "False Negatives (FN)": int(fn),
        "Total Errors (FP+FN)": int(total_errors),
        "AUROC": roc_auc_score(y_test, y_proba),
        "PR AUC": average_precision_score(y_test, y_proba)
    }

    # 4. 데이터프레임으로 변환 후 소수점 포맷팅
    df = pd.DataFrame(list(metrics.items()), columns=["Metric", model_name])

    # 실수형 지표만 소수점 4자리로 포맷팅
    df[model_name] = df[model_name].apply(
        lambda x: f"{x:.4f}" if isinstance(x, float) else x
    )

    return df

In [6]:
# @title **모델 평가 및 기준 표 생성**
comparison_df = evaluate_model(best_rf_model, X_test, y_test, model_name="RF_Optuna")
comparison_df

,Metric,RF_Optuna
0,Accuracy (정확도),1.0000
1,Precision (정밀도),1.0000
2,Recall (재현율),1.0000
3,F1 Score,1.0000
4,False Positives (FP),0.0000
5,False Negatives (FN),0.0000
6,Total Errors (FP+FN),0.0000
7,AUROC,1.0000
8,PR AUC,1.0000


In [7]:
# @title **2. Scikit-Learn Cross Validation & Hyperparametertunning**
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats

# 1. 탐색할 하이퍼파라미터 분포 및 후보군 정의 (Optuna 분포와 동일한 범위)
param_distributions = {
    "n_estimators": [50, 100, 150, 200, 250, 300, 350, 400, 450, 500],
    "max_depth": stats.randint(3, 31),
    "min_samples_split": stats.randint(2, 11),
    "min_samples_leaf": stats.randint(1, 11),
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
    "criterion": ["gini", "entropy", "log_loss"]
}

# 2. 기본 모델 객체 생성
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# 3. RandomizedSearchCV 설정
# n_iter=100은 Optuna의 n_trials=100과 동일하게 100번의 조합을 시도합니다.
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=100,
    cv=3,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

# 4. 탐색 실행
random_search.fit(X_train, y_train)

# 5. 결과 확인 및 최적 모델 추출
print("Best Params:", random_search.best_params_)
print("Best Score:", random_search.best_score_)

random_search.best_estimator_

Best Params: {'bootstrap': True, 'criterion': 'log_loss', 'max_depth': 11, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 8, 'n_estimators': 350}
Best Score: 0.9671361502347416


RandomForestClassifier(criterion='log_loss', max_depth=11, max_features='log2',
                       min_samples_split=8, n_estimators=350, n_jobs=-1,
                       random_state=42)

In [8]:
rf_metrics = evaluate_model(random_search.best_estimator_, X_test, y_test, model_name="RF_Scikit-Learn")
comparison_df["RF_Scikit-Learn"] = rf_metrics["RF_Scikit-Learn"]
comparison_df

,Metric,RF_Optuna,RF_Scikit-Learn
0,Accuracy (정확도),1.0000,0.9510
1,Precision (정밀도),1.0000,0.9107
2,Recall (재현율),1.0000,0.9623
3,F1 Score,1.0000,0.9358
4,False Positives (FP),0.0000,5.0000
5,False Negatives (FN),0.0000,2.0000
6,Total Errors (FP+FN),0.0000,7.0000
7,AUROC,1.0000,0.9818
8,PR AUC,1.0000,0.9861


### **주요 모델 default로 학습하고 예측하기**

In [9]:
!pip install catboost

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from tqdm.auto import tqdm

models = {
    "LogisticRegression": LogisticRegression(random_state=42, max_iter=1000),
    "SVC": SVC(probability=True, random_state=42),
    "KNN": KNeighborsClassifier(n_jobs=-1),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0)
}

for model_name, model in tqdm(list(models.items())):
    print(f'{model_name}학습하고 검증데이터 예측성능')
    model.fit(X_train, y_train)
    model_metrics = evaluate_model(model, X_test, y_test, model_name=model_name)
    comparison_df[model_name] = model_metrics[model_name]

comparison_df

  0%|          | 0/8 [00:00<?, ?it/s]

LogisticRegression학습하고 검증데이터 예측성능


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


SVC학습하고 검증데이터 예측성능
KNN학습하고 검증데이터 예측성능
DecisionTree학습하고 검증데이터 예측성능
GradientBoosting학습하고 검증데이터 예측성능
XGBoost학습하고 검증데이터 예측성능
LightGBM학습하고 검증데이터 예측성능
CatBoost학습하고 검증데이터 예측성능


,Metric,RF_Optuna,RF_Scikit-Learn,LogisticRegression,SVC,KNN,DecisionTree,GradientBoosting,XGBoost,LightGBM,CatBoost
0,Accuracy (정확도),1.0000,0.9510,0.9650,0.9301,0.9231,0.9510,0.9720,0.9580,0.9650,0.9650
1,Precision (정밀도),1.0000,0.9107,0.9444,0.9778,0.9038,0.8966,0.9455,0.9273,0.9444,0.9286
2,Recall (재현율),1.0000,0.9623,0.9623,0.8302,0.8868,0.9811,0.9811,0.9623,0.9623,0.9811
3,F1 Score,1.0000,0.9358,0.9533,0.8980,0.8952,0.9369,0.9630,0.9444,0.9533,0.9541
4,False Positives (FP),0.0000,5.0000,3.0000,1.0000,5.0000,6.0000,3.0000,4.0000,3.0000,4.0000
5,False Negatives (FN),0.0000,2.0000,2.0000,9.0000,6.0000,1.0000,1.0000,2.0000,2.0000,1.0000
6,Total Errors (FP+FN),0.0000,7.0000,5.0000,10.0000,11.0000,7.0000,4.0000,6.0000,5.0000,5.0000
7,AUROC,1.0000,0.9818,0.9945,0.9776,0.9593,0.9572,0.9918,0.9925,0.9916,0.9910
8,PR AUC,1.0000,0.9861,0.9913,0.9690,0.9496,0.8866,0.9912,0.9909,0.9905,0.9904


- **sklearn 토이 데이터셋(iris, wine, breast_cancer 등)마다 랜덤 포레스트를 Optuna(TPE)와 RandomizedSearchCV로 각각 튜닝해서 어느 쪽이 더 잘 나오는지 자동으로 비교**
- **두 방법에 같은 탐색 공간, 같은 시도 횟수, 같은 CV 분할을 주고 train 데이터로만 탐색한 뒤, 보지 않은 test 데이터로 정확도·F1·AUROC를 한 번만 평가해서 데이터 누수를 방지**
- **여러 시드로 반복한 결과를 (데이터셋, 시드) 쌍마다 승/무/패, 평균 차이, Wilcoxon 검정으로 요약하고 CSV로 저장하므로, “Optuna가 항상 우세한가?”를 한 번의 실행으로 확인**

In [12]:
"""
Optuna(TPE) vs scikit-learn RandomizedSearchCV 비교 자동화 (RandomForest 한정)

- 대상: sklearn 토이 분류 데이터셋 (iris, wine, breast_cancer, digits 등)
- 공정한 비교를 위한 조건
    1) 두 방법 모두 train 데이터로만 탐색 (test는 마지막에 한 번만 평가 → 데이터 누수 없음)
    2) 동일한 탐색 공간, 동일한 시도 횟수(n_trials == n_iter)
    3) 동일한 CV 분할(StratifiedKFold)과 동일한 scoring
    4) 여러 시드(분할·탐색 난수)로 반복해서 우연에 의한 승패를 줄임

사용 예) (Colab 셀에서는 main([...]) 형태로 인자를 넘기면 됩니다)
    main(['--trials', '50', '--seeds', '0', '1', '2'])
    python compare_optuna_vs_sklearn.py
    python compare_optuna_vs_sklearn.py --trials 50 --seeds 0 1 2 3 4
    python compare_optuna_vs_sklearn.py --datasets iris wine breast_cancer digits
"""
import argparse
import time
import warnings

import numpy as np
import optuna
import pandas as pd
import scipy.stats as stats
from sklearn.datasets import load_breast_cancer, load_digits, load_iris, load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATASETS = {
    "iris": load_iris,
    "wine": load_wine,
    "breast_cancer": load_breast_cancer,
    "digits": load_digits,
}


# ----------------------------------------------------------------------------
# 탐색 공간 (노트북과 동일한 범위)
# ----------------------------------------------------------------------------
def optuna_params(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
    }


SKLEARN_DISTRIBUTIONS = {
    "n_estimators": list(range(50, 501, 50)),
    "max_depth": stats.randint(3, 31),
    "min_samples_split": stats.randint(2, 11),
    "min_samples_leaf": stats.randint(1, 11),
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
    "criterion": ["gini", "entropy", "log_loss"],
}


# ----------------------------------------------------------------------------
# 평가 (이진/다중 분류 모두 지원)
# ----------------------------------------------------------------------------
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    if y_proba.shape[1] == 2:
        auc = roc_auc_score(y_test, y_proba[:, 1])
    else:
        auc = roc_auc_score(y_test, y_proba, multi_class="ovr", labels=model.classes_)
    return {
        "test_acc": accuracy_score(y_test, y_pred),
        "test_f1_macro": f1_score(y_test, y_pred, average="macro"),
        "test_auroc": auc,
    }


# ----------------------------------------------------------------------------
# 방법별 튜닝
# ----------------------------------------------------------------------------
def tune_optuna(X_train, y_train, cv, n_trials, seed, scoring):
    def objective(trial):
        model = RandomForestClassifier(
            **optuna_params(trial), random_state=seed, n_jobs=-1
        )
        return cross_val_score(model, X_train, y_train, cv=cv, scoring=scoring).mean()

    study = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed)
    )
    study.optimize(objective, n_trials=n_trials)
    best = RandomForestClassifier(**study.best_params, random_state=seed, n_jobs=-1)
    best.fit(X_train, y_train)
    return best, study.best_value, study.best_params


def tune_sklearn(X_train, y_train, cv, n_trials, seed, scoring):
    search = RandomizedSearchCV(
        RandomForestClassifier(random_state=seed, n_jobs=-1),
        param_distributions=SKLEARN_DISTRIBUTIONS,
        n_iter=n_trials,
        cv=cv,
        scoring=scoring,
        random_state=seed,
        n_jobs=1,  # 모델 내부(n_jobs=-1)에서 병렬화 → 두 방법의 병렬 조건을 동일하게
        refit=True,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_score_, search.best_params_


# ----------------------------------------------------------------------------
# 메인 루프
# ----------------------------------------------------------------------------
def run(datasets, seeds, n_trials, n_splits, scoring):
    rows = []
    for ds_name in datasets:
        data = DATASETS[ds_name]()
        X, y = data.data, data.target
        for seed in seeds:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.25, stratify=y, random_state=seed
            )
            # 두 방법이 완전히 같은 폴드를 쓰도록 동일한 cv 객체 사용
            cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

            for method, tuner in (("optuna", tune_optuna), ("sklearn", tune_sklearn)):
                t0 = time.perf_counter()
                model, cv_score, best_params = tuner(
                    X_train, y_train, cv, n_trials, seed, scoring
                )
                elapsed = time.perf_counter() - t0
                rows.append(
                    {
                        "dataset": ds_name,
                        "seed": seed,
                        "method": method,
                        "cv_score": cv_score,
                        **evaluate(model, X_test, y_test),
                        "time_sec": elapsed,
                        "best_params": best_params,
                    }
                )
            print(f"[done] {ds_name} seed={seed}", flush=True)
    return pd.DataFrame(rows)


def summarize(df, metric="test_acc", tol=1e-9):
    """(dataset, seed) 쌍 단위로 optuna - sklearn 차이를 계산해 승/무/패 집계."""
    wide = df.pivot_table(index=["dataset", "seed"], columns="method", values=metric)
    wide["diff"] = wide["optuna"] - wide["sklearn"]

    def wtl(d):
        return pd.Series(
            {
                "optuna_win": int((d > tol).sum()),
                "tie": int((d.abs() <= tol).sum()),
                "sklearn_win": int((d < -tol).sum()),
                "mean_diff": d.mean(),
                "std_diff": d.std(ddof=1) if len(d) > 1 else np.nan,
            }
        )

    per_ds = wide.groupby("dataset")["diff"].apply(wtl).unstack()
    overall = wtl(wide["diff"])

    # 전체 쌍에 대한 Wilcoxon 부호순위 검정 (차이가 전부 0이면 불가)
    try:
        p = stats.wilcoxon(wide["diff"][wide["diff"].abs() > tol]).pvalue
    except ValueError:
        p = np.nan
    return per_ds, overall, p


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--datasets", nargs="+", default=["iris", "wine", "breast_cancer"],
                    choices=list(DATASETS))
    ap.add_argument("--seeds", nargs="+", type=int, default=[0, 1, 2, 3, 4])
    ap.add_argument("--trials", type=int, default=100)
    ap.add_argument("--cv", type=int, default=3)
    ap.add_argument("--scoring", default="accuracy")
    ap.add_argument("--out", default="optuna_vs_sklearn_results.csv")
    # Jupyter/Colab이 붙이는 '-f kernel.json' 같은 인자는 무시
    args, _ = ap.parse_known_args(argv)

    df = run(args.datasets, args.seeds, args.trials, args.cv, args.scoring)
    df.drop(columns="best_params").to_csv(args.out, index=False)
    df[["dataset", "seed", "method", "best_params"]].to_csv(
        args.out.replace(".csv", "_params.csv"), index=False
    )

    pd.set_option("display.width", 200)
    print("\n===== 데이터셋·방법별 평균 (시드 평균) =====")
    print(
        df.groupby(["dataset", "method"])[
            ["cv_score", "test_acc", "test_f1_macro", "test_auroc", "time_sec"]
        ].mean().round(4)
    )

    for metric in ("test_acc", "test_f1_macro", "test_auroc"):
        per_ds, overall, p = summarize(df, metric)
        print(f"\n===== [{metric}] optuna - sklearn 차이 (양수면 Optuna 우세) =====")
        print(per_ds.round(4))
        print("전체:", overall.round(4).to_dict(), f"| Wilcoxon p={p:.4f}" if not np.isnan(p) else "| Wilcoxon 불가(차이 모두 0)")

    print(f"\n결과 저장: {args.out}")


if __name__ == "__main__":
    main()

[done] iris seed=0
[done] iris seed=1
[done] iris seed=2
[done] iris seed=3
[done] iris seed=4
[done] wine seed=0
[done] wine seed=1
[done] wine seed=2
[done] wine seed=3
[done] wine seed=4
[done] breast_cancer seed=0
[done] breast_cancer seed=1
[done] breast_cancer seed=2
[done] breast_cancer seed=3
[done] breast_cancer seed=4

===== 데이터셋·방법별 평균 (시드 평균) =====
                       cv_score  test_acc  test_f1_macro  test_auroc  time_sec
dataset       method                                                          
breast_cancer optuna     0.9662    0.9636         0.9607      0.9878  310.6125
              sklearn    0.9638    0.9622         0.9592      0.9871  292.9888
iris          optuna     0.9517    0.9632         0.9626      0.9879  217.8282
              sklearn    0.9553    0.9579         0.9574      0.9939  201.9435
wine          optuna     0.9699    0.9867         0.9877      0.9999  209.9319
              sklearn    0.9730    0.9911         0.9918      1.0000  202.2465

====

In [18]:
target_files

['optuna_vs_sklearn_results.csv', 'optuna_vs_sklearn_results_params.csv']

In [19]:
pd.read_csv('optuna_vs_sklearn_results.csv')

,dataset,seed,method,cv_score,test_acc,test_f1_macro,test_auroc,time_sec
0,iris,0,optuna,0.955429,0.947368,0.945887,0.971068,187.910475
1,iris,0,sklearn,0.955429,0.973684,0.973162,1.000000,215.859997
2,iris,1,optuna,0.928402,0.973684,0.974321,0.995897,137.219466
3,iris,1,sklearn,0.937174,0.973684,0.974321,0.995897,192.077016
4,iris,2,optuna,0.964438,1.000000,1.000000,1.000000,260.830782
5,iris,2,sklearn,0.964438,1.000000,1.000000,1.000000,197.990452
6,iris,3,optuna,0.982219,0.921053,0.919485,0.983248,286.755065
7,iris,3,sklearn,0.982456,0.894737,0.893162,0.985342,190.086432
8,iris,4,optuna,0.928165,0.973684,0.973162,0.989530,216.425201
9,iris,4,sklearn,0.937174,0.947368,0.946581,0.988462,213.703600


In [20]:
pd.read_csv('optuna_vs_sklearn_results_params.csv')

,dataset,seed,method,best_params
0,iris,0,optuna,"{'n_estimators': 350, 'max_depth': 20, 'min_sa..."
1,iris,0,sklearn,"{'bootstrap': True, 'criterion': 'entropy', 'm..."
2,iris,1,optuna,"{'n_estimators': 200, 'max_depth': 28, 'min_sa..."
3,iris,1,sklearn,"{'bootstrap': False, 'criterion': 'gini', 'max..."
4,iris,2,optuna,"{'n_estimators': 300, 'max_depth': 12, 'min_sa..."
5,iris,2,sklearn,"{'bootstrap': True, 'criterion': 'gini', 'max_..."
6,iris,3,optuna,"{'n_estimators': 400, 'max_depth': 10, 'min_sa..."
7,iris,3,sklearn,"{'bootstrap': True, 'criterion': 'log_loss', '..."
8,iris,4,optuna,"{'n_estimators': 450, 'max_depth': 30, 'min_sa..."
9,iris,4,sklearn,"{'bootstrap': False, 'criterion': 'entropy', '..."


**RandomForest 모델과 토이 데이터셋(iris, wine, breast_cancer) 환경에서 Optuna와 scikit-learn(RandomizedSearchCV)의 성능(test_acc, test_f1_macro, test_auroc)은 전반적으로 매우 비슷하며 큰 차이가 없음**

In [13]:
import os
import zipfile
from google.colab import files

# 1. 저장된 결과 파일 목록 (기본 파일명 기준)
target_files = [
    "optuna_vs_sklearn_results.csv",
    "optuna_vs_sklearn_results_params.csv",
]

# 2. 두 파일을 하나의 zip으로 묶어서 한 번에 다운로드 (브라우저 다중 다운로드 차단 방지)
zip_filename = "optuna_vs_sklearn_output.zip"

with zipfile.ZipFile(zip_filename, "w") as zf:
    for f in target_files:
        if os.path.exists(f):
            zf.write(f)
            print(f"압축 포함 완료: {f}")
        else:
            print(f"[주의] 파일을 찾을 수 없습니다: {f}")

if os.path.exists(zip_filename):
    files.download(zip_filename)

압축 포함 완료: optuna_vs_sklearn_results.csv
압축 포함 완료: optuna_vs_sklearn_results_params.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>